In [0]:
from pyspark.sql.functions import to_date, col, when
from datetime import datetime
import requests
import csv
import time
import json

In [0]:
# Authenticating Azure Data Lake Storage Gen2
storage_account = "**********************"

# get storage access key
storage_key = "*****************************************************************************************"

# configure databricks to access storage account
spark.conf.set(
  f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
  storage_key
)

BRONZE = f"abfss://********************@{storage_account}.dfs.core.windows.net/"
DATA_TRACKING = f"abfss://*************************@{storage_account}.dfs.core.windows.net/"

In [0]:
API_KEY = "****************"
API_KEY1 = "*****************"
tickers = ["AAPL", "MSFT", "AMZN", "GOOGL", "NVDA", "TSLA", "META", "BRK.B", "JPM", "V"]

def fetch_stock_data(tickers):
    rows = []

    for symbol in tickers:
        url1 = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={API_KEY}'
        url = "https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=IBM&apikey=demo"

        response = requests.get(url, timeout=10)
        data = response.json()

        # CHECK FIRST (before ANY access)
        if "Time Series (Daily)" not in data:
            print(f"Error with {symbol}")
            print(data)   # important debug
            continue

        for date, values in data["Time Series (Daily)"].items():
            rows.append({
                "symbol": symbol,
                "date": date,
                **values
            })
        time.sleep(3)

    return rows

In [0]:
# Converting API data rows to Spark DataFrame
rows = fetch_stock_data(tickers)
df = spark.createDataFrame(rows)
display(df)

In [0]:
# Casting the date column to date type
df = df.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

In [0]:
# Getting the last stock date from latest stock date ADLS
last_date_df = spark.read.option("multiline", "true").json(f'{DATA_TRACKING}/bronze_last_date.json')

# Getting the actual last date value
last_date = last_date_df.where(col("1-pipeline") == "bronze").select("2-latest_date").first()[0]
#### last_date = "2026-04-22"

# Getting the actual ealier date value
#### early_date = last_date_df.where(col("1-pipeline") == "bronze").select("3-earlier_date").first()[0]

# Converting to date type
last_date = datetime.strptime(last_date, "%Y-%m-%d").date()
#### early_date = datetime.strptime(early_date, "%Y-%m-%d").date()

print(last_date)
#### print(early_date)

In [0]:
# USE THIS SECTION FOR TESTING
#######################   # df_filtered = df.filter(col("date") > early_date).filter(col("date") <= last_date)
#######################   df_filtered = df.filter((col("date") > early_date) & (col("date") <= last_date))
#######################   display(df_filtered)

# USE THIS SECTION IN PRODUCTION
# Filtering only rows greater than the lastest date in our watermark
df_filtered = df.filter(col("date") > last_date)
display(df_filtered)



In [0]:
# Getting the latest stock date to update our watermark in ADLS
new_last_date = df.orderBy(col("date").desc()).select("date").first()[0]

# Updating and overwriting the latest stock date to ADLS
bronze_last_date = {
    "1-pipeline": "bronze",
    "2-latest_date": new_last_date,
    "3-earlier_date": last_date
}

file_path = f"{DATA_TRACKING}/bronze_last_date.json"

# Save the JSON data
json_data = json.dumps(bronze_last_date, indent=4, default=str)
dbutils.fs.put(file_path, json_data, overwrite=True)

In [0]:
# Save the transformed DataFrame to the Bronze container
bronze_output_path = f"{BRONZE}bronze_capitaledge/"

# Append DataFrame to Bronze container in Parquet format
df_filtered.write.mode('append').parquet(bronze_output_path)